<h2> Classify spam vs no spam emails</h2>

In [ ]:
!pip install tensorflow_hub

In [ ]:
!pip install tensorflow_text

In [ ]:
!pip install tensorflow

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text

<h4>Importer la dataset </h4>

In [ ]:
import pandas as pd

df = pd.read_csv("spam.csv")
df.head(5)

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df.groupby('Category').describe()

Message                                                            \
           count unique                                                top   
Category                                                                     
ham         4825   4516                             Sorry, I'll call later   
spam         747    641  Please call our customer service representativ...   

               
         freq  
Category       
ham        30  
spam        4

In [ ]:
df['spam']=df['Category'].apply(lambda x: 1 if x=='spam' else 0)
df.head()

,Category,Message,spam
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


<h4>On dévise la data en train et test data </h4>

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['Message'],df['spam'], stratify=df['spam'])

In [ ]:
X_train.head(4)

,Message
1058,Ard 515 like dat. Y?
3213,We got a divorce. Lol. She.s here
360,"Hello! Just got here, st andrews-boy its a lon..."
55,Do you know what Mallika Sherawat did yesterda...


<h4>ici on essayer d'appeller le tekonizer zt l ' encoudeure </h4>

In [ ]:
bert_preprocess_model = "https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3"
bert_model = "https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4"

In [ ]:
bert_preprocess = hub.KerasLayer(bert_preprocess_model)
bert_encoder = hub.KerasLayer(bert_model)


In [ ]:
def get_sentence_embeding(sentences):
    preprocessed_text = bert_preprocess(sentences)
    return bert_encoder(preprocessed_text)['pooled_output']

get_sentence_embeding([
    "500$ discount. hurry up",
    "Bhavin, are you up for a volleybal game tomorrow?"]
)

<tf.Tensor: shape=(2, 768), dtype=float32, numpy=
array([[-0.8435168 , -0.5132724 , -0.8884571 , ..., -0.7474884 ,
        -0.7531473 ,  0.91964483],
       [-0.87208354, -0.50543964, -0.94446665, ..., -0.85847497,
        -0.7174535 ,  0.88082975]], dtype=float32)>

<h4>Obtenir les vecteurs d'embedding pour quelques mots d'exemple. Les comparer à l'aide de la similarité cosinus</h4>


In [ ]:
e = get_sentence_embeding([
    "banana",
    "grapes",
    "mango",
    "jeff bezos",
    "elon musk",
    "bill gates"
]
)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_similarity([e[0]],[e[1]])

array([[0.9911088]], dtype=float32)

Des valeurs proches de 1 signifient que les éléments sont similaires. Une valeur proche de 0 signifie qu'ils sont très différents.
Par exemple, en comparant "banane" et "raisin", on obtient une similarité de 0,99 car ce sont tous les deux des fruits.

In [ ]:
cosine_similarity([e[0]],[e[3]])

array([[0.84703827]], dtype=float32)

In [ ]:
cosine_similarity([e[3]],[e[4]])

array([[0.9872036]], dtype=float32)

<h4>Build Model</h4>

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text  # IMPORTANT : à ne pas oublier

# URLs des modèles
bert_preprocess_url = "https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3"
bert_encoder_url = "https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4"

# Charger les couches
bert_preprocess_layer = hub.KerasLayer(bert_preprocess_url, name="preprocessing", trainable=False)
bert_encoder_layer = hub.KerasLayer(bert_encoder_url, name="BERT_encoder", trainable=True)

# Subclassed model
class BERTClassifier(tf.keras.Model):
    def __init__(self, encoder, preprocess):
        super(BERTClassifier, self).__init__()
        self.preprocess = preprocess
        self.encoder = encoder
        self.dropout = tf.keras.layers.Dropout(0.1)
        self.classifier = tf.keras.layers.Dense(1, activation='sigmoid')

    def call(self, inputs):
        x = self.preprocess(inputs)
        x = self.encoder(x)['pooled_output']
        x = self.dropout(x)
        return self.classifier(x)

# Créer le modèle
model = BERTClassifier(bert_encoder_layer, bert_preprocess_layer)


In [ ]:
len(X_train)

4179

In [ ]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

<h4>Train the model</h4>

In [ ]:
X_train = tf.convert_to_tensor(X_train)
y_train = tf.convert_to_tensor(y_train)


In [ ]:
model.fit(X_train, y_train, epochs=5)

Epoch 1/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 2196s 17s/step - accuracy: 0.7516 - loss: 0.5237
Epoch 2/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 2126s 16s/step - accuracy: 0.8635 - loss: 0.2883
Epoch 3/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 2160s 16s/step - accuracy: 0.8935 - loss: 0.2281
Epoch 4/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 2115s 16s/step - accuracy: 0.9283 - loss: 0.1949
Epoch 5/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 2095s 16s/step - accuracy: 0.9313 - loss: 0.1814


In [ ]:
X_test = tf.convert_to_tensor(X_test)
y_test = tf.convert_to_tensor(y_test)


In [ ]:
model.evaluate(X_test, y_test)

44/44 ━━━━━━━━━━━━━━━━━━━━ 694s 16s/step - accuracy: 0.9512 - loss: 0.1696


[0.15959131717681885, 0.9583632349967957]

<h4>Inference</h4>

In [ ]:
reviews = [
    'Reply to win Â£100 weekly! Where will the 2006 FIFA World Cup be held? Send STOP to 87239 to end service',
    'You are awarded a SiPix Digital Camera! call 09061221061 from landline. Delivery within 28days. T Cs Box177. M221BP. 2yr warranty. 150ppm. 16 . p pÂ£3.99',
    'it to 80488. Your 500 free text messages are valid until 31 December 2005.',
    'Hey Sam, Are you coming for a cricket game tomorrow',
    "Why don't you wait 'til at least wednesday to see if you get your ."
]
reviews = tf.convert_to_tensor(reviews)

model.predict(reviews)

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step


array([[0.5997449 ],
       [0.6462637 ],
       [0.51966745],
       [0.06311313],
       [0.03162586]], dtype=float32)

In [ ]:
!pip install gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.3 MB/s eta 0:00:00


In [ ]:
import gradio as gr

# Exemple de fonction prédictive, ici tu peux brancher ton modèle réel
def predict_spam(text):
    result = model.predict(tf.convert_to_tensor(text))
    return "Spam" if result == 1 else "Non Spam"

# Interface Gradio
interface = gr.Interface(
    fn=predict_spam,
    inputs=gr.Textbox(lines=5, placeholder="Entrez un message..."),
    outputs="text",
    title="Détecteur de Spam",
    description="Entrez un message pour savoir s'il s'agit d'un spam ou non."
)

interface.launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e5f88626228ba2e4b2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
